# Notebook 02 — Training the from-scratch Transformer encoder

We train **our own** Transformer text-encoder (no pretrained model, no
`transformers` / `sentence-transformers`) with the InfoNCE contrastive loss.

**Steps**
1. Train a from-scratch **BPE tokenizer** (`src/tokenizer.py`) on the corpus + training pairs.
2. Build the **Transformer encoder** (`src/model.py`): token embeddings + sinusoidal
   positional encoding + multi-head self-attention + feed-forward, pre-norm residuals,
   masked mean pooling, L2 normalization.
3. Train it with **symmetric InfoNCE** (`src/loss.py`) using in-batch negatives.
4. Plot train/val loss curves and save the checkpoint + tokenizer to Drive.

## Training setup
| Hyperparameter | Value | Why |
|---|---|---|
| Tokenizer | BPE, vocab 8000 | learned from our corpus; good sub-word coverage of technical terms |
| d_model / layers / heads | 256 / 4 / 4 | ~5M params — trains fast on a T4 |
| Loss | symmetric InfoNCE | in-batch negatives: batch 64 → 63 free negatives per query |
| Temperature | 0.05 | sharper similarity distribution; helps from-scratch training |
| Batch size | 64 | more negatives per step = stronger contrastive signal |
| Learning rate | 3e-4 | training **from scratch** needs a higher LR than fine-tuning (2e-5) |
| LR schedule | linear warmup (10%) + linear decay | stabilizes the first, noisiest steps |
| Epochs | 20 | randomly-initialized weights need more passes than a fine-tuned model |
| Doc / query max length | 256 / 64 | documents are long (~255 words), queries are short |
| Gradient clipping | 1.0 | guards against early gradient spikes |

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point this at wherever you uploaded the project on your Drive:
    PROJECT_ROOT = '/content/drive/MyDrive/NLP PROJECT/Neural_Search_Engine-main'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    # The from-scratch model only needs torch (already installed in Colab).
    # No transformers / tokenizers / safetensors / sentence-transformers.
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

import torch
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))
    print('VRAM     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import json

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

# Search corpus (what the engine retrieves from) + contrastive training pairs.
chunks      = load_json('data/processed/jurafsky_chunks_v2.json')      # 1410 Jurafsky chunks
train_pairs = load_json('data/processed/train_pairs_combined.json')    # Jurafsky + Wikipedia
val_pairs   = load_json('data/processed/val_pairs_combined.json')

print(f'Corpus chunks : {len(chunks)}')
print(f'Train pairs   : {len(train_pairs)}')
print(f'Val pairs     : {len(val_pairs)}')

In [3]:
p = train_pairs[0]
print(f"Example query    : {p['query']}")
print(f"Example positive : {p['positive_text'][:120]}")
print(f"Example negative : {p['negative_text'][:120]}")

Example query    : What historical privacy concerns were raised during early experiments with Joseph Weizenbaum's therapeutic ELIZA chatbot?
Example positive : They found, for example, that simple changes like using the word ‘she’ instead of ‘he’ in a sentence caused systems to r
Example negative : This arrangement correctly permits imagine a hamburger and lift a hamburger, while also correctly ruling out diagonalize


In [ ]:
from src.model import BiEncoder, EncoderConfig
from src.train import Trainer

# --- Hyperparameters (see the table at the top) ---
D_MODEL       = 256
N_LAYERS      = 4
N_HEADS       = 4
DOC_MAX_LEN   = 256   # documents (chunks) are long
QUERY_MAX_LEN = 64    # queries are short
BATCH_SIZE    = 64
EPOCHS        = 20
LR            = 3e-4
TEMPERATURE   = 0.05
CHECKPOINT    = 'checkpoints'

config = EncoderConfig(
    vocab_size = tokenizer.vocab_size,
    d_model    = D_MODEL,
    n_layers   = N_LAYERS,
    n_heads    = N_HEADS,
    d_ff       = D_MODEL * 4,
    max_len    = DOC_MAX_LEN,
)
model = BiEncoder(config)
print(f'Model parameters: {model.num_parameters():,}')

In [ ]:
trainer = Trainer(
    model         = model,
    tokenizer     = tokenizer,
    train_pairs   = train_pairs,
    val_pairs     = val_pairs,
    output_dir    = CHECKPOINT,
    query_max_len = QUERY_MAX_LEN,
    doc_max_len   = DOC_MAX_LEN,
    batch_size    = BATCH_SIZE,
    epochs        = EPOCHS,
    lr            = LR,
    temperature   = TEMPERATURE,
)

print('Trainer ready.')
print(f'Batches per epoch : {len(trainer.train_loader)}')
print(f'Total steps       : {len(trainer.train_loader) * EPOCHS}')

In [ ]:
import time

t0 = time.time()
history = trainer.fit()
print(f'\nTotal training time: {(time.time() - t0) / 60:.1f} min')

In [ ]:
import math
import matplotlib.pyplot as plt

epochs_x = list(range(1, len(history['train_loss']) + 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_x, history['train_loss'], marker='o', label='Train loss', color='steelblue')
ax.plot(epochs_x, history['val_loss'],   marker='s', label='Val loss',   color='tomato')
ax.axhline(y=min(history['val_loss']), linestyle='--', color='tomato', alpha=0.4)

# An untrained model can't tell the positive from the in-batch negatives, so its
# loss sits around log(batch_size). Watching the curve fall well below this line
# is direct evidence the encoder is learning to match queries to documents.
random_loss = math.log(BATCH_SIZE)
ax.axhline(y=random_loss, linestyle=':', color='grey', alpha=0.6,
           label=f'Random baseline log(B)={random_loss:.2f}')

ax.set_xlabel('Epoch')
ax.set_ylabel('Symmetric InfoNCE loss')
ax.set_title('From-scratch encoder — training curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150)
plt.show()

print(f'Best val loss : {min(history["val_loss"]):.4f}')
print(f'Random loss   : {random_loss:.4f}   (untrained reference)')
print(f'Improvement   : {random_loss - min(history["val_loss"]):.4f}')

In [ ]:
import pandas as pd

log_df = pd.read_csv('checkpoints/training_log.csv')
print(log_df.to_string(index=False))

In [ ]:
# What we saved (model checkpoints carry their config; tokenizer travels alongside).
for fname in sorted(os.listdir('checkpoints')):
    fpath = os.path.join('checkpoints', fname)
    if os.path.isfile(fpath):
        print(f'  {fname:<24} {os.path.getsize(fpath) / 1e6:.2f} MB')

# Sanity check: reload the best checkpoint and confirm it rebuilds.
from src.model import BiEncoder

model_loaded = BiEncoder.load('checkpoints/best.pt')
print('\nReloaded best.pt — parameters:', f'{model_loaded.num_parameters():,}')

In [ ]:
import os
for fname in os.listdir('checkpoints'):
    fpath = os.path.join('checkpoints', fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<30} {size_mb:.1f} MB')

model_loaded = BiEncoder()
model_loaded.load_state_dict(torch.load('checkpoints/best.pt', map_location='cpu'))
model_loaded.eval()